# LLM API Tutorial (OpenAI, Anthropic, Gemini)

This tutorial demonstrates the **common structure** for using modern LLM APIs.

We will progressively build:

1. Basic API call
2. Add Configuration


## 0. Setup Environment

## 1. Install Dependencies

# 2. Environment Variables

Create a `.env` file:


```env
OPENAI_API_KEY=your_openai_key
ANTHROPIC_API_KEY=your_anthropic_key
GEMINI_API_KEY=your_google_key
```

## 3. Common API Pattern

Use this mental model for every provider:

```text
client
   handles connection

model
   chooses capability

system prompt
   defines behavior

tools
   gives abilities

config
   tunes generation

user question
   defines task
```


Recommended structure:

```python
# 1. Client
client = ProviderClient()

# 2. Model
MODEL = "..."

# 3. Default prompt
SYSTEM_PROMPT = "You are a helpful assistant."

# 4. Optional tools
TOOLS = [...]

# 5. Optional generation settings
CONFIG = {...}

# 6. Dynamic user question
USER_QUESTION = "..."

# 7. API request
response = ...
```

## Part 1 — OpenAI

Official docs: https://platform.openai.com/docs

### 1.1 Basic API Call

In [4]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# -------------------
# 1. Client
# -------------------
client = OpenAI()

# -------------------
# 2. Model
# -------------------
MODEL = "gpt-4o-mini"

# -------------------
# 3. System Prompt
# -------------------
SYSTEM_PROMPT = "You are a helpful assistant."

# -------------------
# 4. Tools
# -------------------
TOOLS = []

# -------------------
# 5. Config
# -------------------
CONFIG = {

}

# -------------------
# 6. User Question
# -------------------
USER_QUESTION = "Explain what an LLM is."

# -------------------
# 7. Request
# -------------------
response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    tools=TOOLS,
    input=USER_QUESTION,
    **CONFIG
)

print(response.output_text)

An LLM, or Large Language Model, is a type of artificial intelligence model designed to understand and generate human language. These models are trained on vast amounts of text data from the internet, books, articles, and other written sources. 

Key characteristics of LLMs include:

1. **Scale**: They have millions or even billions of parameters, which are the weights adjusted during training to help the model make predictions and generate text.

2. **Training**: LLMs typically use a technique called unsupervised learning, where they learn patterns and structures in language without explicit annotations. They often undergo fine-tuning on specific tasks or domains afterwards.

3. **Capabilities**: They can perform a variety of language tasks, such as:
   - Text generation
   - Translation
   - Summarization
   - Question answering
   - Conversational interaction

4. **Architecture**: Most LLMs are based on transformer architecture, which utilizes mechanisms like self-attention to proce

Configurations:

#### `temperature`

| Value  | Behavior                |
| ------ | ----------------------- |
| `0.0`  | deterministic           |
| `0.2`  | focused                 |
| `0.7`  | balanced (good default) |
| `1.0`  | creative                |
| `>1.2` | more random             |

#### `max_output_token`

| Task            | Suggested   |
| --------------- | ----------- |
| Short answers   | `200`       |
| Explanation     | `500`       |
| Long tutorial   | `1000–2000` |
| Code generation | `1500+`     |


In [5]:
client = OpenAI()

MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = "You are a helpful assistant."

TOOLS = [
]

CONFIG = {
    "temperature": 0.2,
    "max_output_tokens": 500,
}

USER_QUESTION = "Explain how LLM distillation works."

response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    tools=TOOLS,
    input=USER_QUESTION,
    **CONFIG
)


print(response.output_text)


LLM distillation, or Language Model Distillation, is a process used to create a smaller, more efficient model from a larger, more complex language model (LLM). The goal is to retain much of the performance of the larger model while reducing its size and computational requirements. Here’s how it generally works:

### 1. **Teacher-Student Framework**
   - **Teacher Model**: The larger, pre-trained language model serves as the "teacher." It has been trained on vast amounts of data and has learned to generate high-quality text.
   - **Student Model**: The smaller model, which is the "student," is typically a simpler architecture or has fewer parameters.

### 2. **Training Process**
   - **Data Preparation**: The training data used for the teacher model is often reused. The student model will learn from the outputs of the teacher model rather than the raw input data.
   - **Soft Targets**: Instead of using hard labels (like the correct next word), the student model learns from the teacher's

### ChatGPT API Changes for GPT-5

Here’s a **clean, accurate summary of GPT-4 vs GPT-5 API changes** (based on the latest Responses API design and GPT-5 docs).

---

### 🧠 GPT-4 vs GPT-5 API — Key Differences

#### 1. Core design shift

##### GPT-4 family (gpt-4o, gpt-4o-mini)

```text
Goal: generate text
Control: randomness
```

You control behavior with:

* `temperature`
* `top_p`
* `max_tokens`

---

##### GPT-5 family (gpt-5, gpt-5-nano, etc.)

```text
Goal: reason + generate
Control: thinking effort + verbosity
```

You control behavior with:

* `reasoning.effort`
* `text.verbosity`
* `max_output_tokens`

📌 Source: GPT-5 API introduces reasoning + verbosity controls ([OpenAI][1])

---

### 2. Parameter changes (IMPORTANT)

#### ❌ Removed / not used the same way

##### GPT-5 does NOT rely on:

* `temperature` (not primary control anymore for GPT-5 reasoning models)
* `top_p` (rarely used in GPT-5 reasoning flow)

👉 GPT-5 shifts away from randomness tuning.

---

#### ✅ New GPT-5 parameters

##### 1. Reasoning control (NEW)

```python
"reasoning": {
    "effort": "minimal | low | medium | high"
}
```

Think:

```text
How hard should the model think?
```

📌 GPT-5 supports configurable reasoning effort levels ([OpenAI][1])

---

##### 2. Output style control (NEW)

```python
"text": {
    "verbosity": "low | medium | high"
}
```

Think:

```text
How detailed should the answer be?
```

📌 GPT-5 adds verbosity control for response length and detail ([OpenAI][1])

---

##### 3. Output length (same concept)

```python
"max_output_tokens": 500
```

Still controls hard token limit.

---

### 3. API structure change

#### GPT-4 style (classic)

```python
response = client.responses.create(
    model="gpt-4o-mini",
    input="Hello",
    temperature=0.7,
    max_output_tokens=500
)
```

---

#### GPT-5 style (modern Responses API)

```python
response = client.responses.create(
    model="gpt-5-nano",
    input="Hello",
    reasoning={
        "effort": "minimal"
    },
    text={
        "verbosity": "medium"
    },
    max_output_tokens=500
)
```

---

### 4. Mental model shift (VERY IMPORTANT)

#### GPT-4 thinking

```text
“Should I be more random or deterministic?”
```

→ temperature-driven

---

#### GPT-5 thinking

```text
“How much should I reason before answering?”
“How detailed should the output be?”
```

→ reasoning-driven

---

### 5. Model differences (API perspective)

#### GPT-4 models

* gpt-4o
* gpt-4o-mini

Characteristics:

* fast
* flexible
* simpler control
* strong chat behavior

Best for:

* chatbots
* summarization
* general assistants

---

#### GPT-5 models

* gpt-5
* gpt-5-mini
* gpt-5-nano

Characteristics:

* explicit reasoning control
* better multi-step reasoning
* structured output alignment improved
* more predictable “thinking level”

📌 GPT-5 is optimized for coding + agentic + reasoning tasks ([OpenAI][1])

---

### 6. Tool calling differences

#### GPT-4

```python
tools=[{"type": "function"}]
```

---

#### GPT-5

Same concept but:

* more reliable tool selection
* supports custom tools + built-in tools better
* reasoning happens before tool calls (more structured)

---

### 7. Key practical differences (summary table)

| Feature           | GPT-4 (4o / 4o-mini) | GPT-5 (gpt-5 / nano)        |
| ----------------- | -------------------- | --------------------------- |
| Primary control   | temperature          | reasoning.effort            |
| Output control    | implicit             | verbosity                   |
| API style         | simpler              | structured config objects   |
| Thinking behavior | hidden               | explicit                    |
| Best use          | chat + general tasks | reasoning + agents + coding |
| Predictability    | medium               | high                        |

---

### 8. Final “proper syntax” templates

## GPT-4o / GPT-4o-mini (classic)

```python
CONFIG = {
    "temperature": 0.7,
    "max_output_tokens": 500
}
```

---

#### GPT-5 / GPT-5-nano (modern)

```python
CONFIG = {
    "reasoning": {
        "effort": "minimal"
    },
    "text": {
        "verbosity": "medium"
    },
    "max_output_tokens": 500
}
```

---

### 9. Key takeaway (important)

##### GPT-4 mindset:

> “Control randomness”

##### GPT-5 mindset:

> “Control thinking depth + output style”



In [7]:
client = OpenAI()

MODEL = "gpt-5-nano"

SYSTEM_PROMPT = "You are a helpful assistant."

TOOLS = []

CONFIG = {
    "reasoning": {
        "effort": "low"
    },
    "text": {
        "verbosity": "medium"
    },
    "max_output_tokens": 500
}

USER_QUESTION = "Explain how LSTMs work."

response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    tools=TOOLS,
    input=USER_QUESTION,
    **CONFIG
)

print(response.output_text)

Long short-term memory networks (LSTMs) are a type of recurrent neural network (RNN) designed to learn long-range dependencies in sequence data. They address the vanishing/exploding gradient problems that make standard RNNs hard to train on long sequences.

Core ideas
- They maintain a hidden state that flows through time, but with a special memory mechanism (cell state) that can carry information across many time steps.
- They use gates to control what information is kept, forgotten, or added to the cell state. This gating makes the network learn what to remember and for how long.

Key components at each time step t
- Input x_t: the data at time t.
- Hidden state h_t: the output of the LSTM at time t (also used to compute outputs and passed to the next step).
- Cell state c_t: the “memory” of the network that can carry information across many time steps.

Gates and their roles
1. Forget gate f_t
   - Decides what information to discard from the previous cell state c_{t-1}.
   - f_t = 


# Part 2 — Anthropic Claude

Official docs: https://docs.anthropic.com

## 2.1 Basic API Call


In [9]:
import anthropic
from dotenv import load_dotenv

load_dotenv()

# -------------------
# 1. Client
# -------------------
client = anthropic.Anthropic()

# -------------------
# 2. Model
# -------------------
MODEL = "claude-sonnet-4-5"

# -------------------
# 3. System Prompt
# -------------------
SYSTEM_PROMPT = "You are a helpful assistant."

# -------------------
# 4. Tools
# -------------------
TOOLS = []

# -------------------
# 5. Config
# -------------------
CONFIG = {
    "max_tokens": 500,
    "temperature": 0.2
}

# -------------------
# 6. User Question
# -------------------
USER_QUESTION = "Explain transformers simply."

# -------------------
# 7. Request
# -------------------
response = client.messages.create(
    model=MODEL,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": USER_QUESTION
        }
    ],
    **CONFIG
)

print(response.content[0].text)


# Transformers Explained Simply

**Transformers** are a type of AI model that processes information by paying attention to different parts of the input at once.

## The Key Innovation: Attention

Think of reading a sentence. To understand a word, you naturally look at the words around it. Transformers do this mathematically through **"attention"** - they learn which parts of the input are most relevant to each other.

## How They Work (Basic Steps)

1. **Input**: Text is converted into numbers (tokens)
2. **Attention**: The model looks at all words simultaneously and figures out relationships between them
3. **Processing**: Multiple layers refine the understanding
4. **Output**: Generate predictions (next word, translation, answer, etc.)

## Why They're Powerful

- **Parallel processing**: Unlike older models that read word-by-word, transformers process everything at once (much faster)
- **Long-range connections**: They can connect ideas far apart in text
- **Flexible**: Same architect

# Part 3 — Google Gemini

Official docs: https://ai.google.dev/gemini-api/docs


## 3.1 Basic API Call

In [10]:
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

# -------------------
# 1. Client
# -------------------
client = genai.Client()

# -------------------
# 2. Model
# -------------------
MODEL = "gemini-2.5-flash-lite"

# -------------------
# 3. System Prompt
# -------------------
SYSTEM_PROMPT = "You are a helpful assistant."

# -------------------
# 4. Config
# -------------------
CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=0.2
)

# -------------------
# 5. User Question
# -------------------
USER_QUESTION = "Explain embeddings simply."

# -------------------
# 6. Request
# -------------------
response = client.models.generate_content(
    model=MODEL,
    contents=USER_QUESTION,
    config=CONFIG
)

print(response.text)


Imagine you have a bunch of words, like "apple," "banana," "car," and "truck."

**Embeddings are like giving each of these words a unique "address" in a special kind of space.**

This space isn't like a regular map with north, south, east, and west. Instead, it has many "dimensions" (think of them as different directions).

Here's the key idea:

*   **Similar words get similar addresses.** Words that mean similar things will have addresses that are close to each other in this space. So, "apple" and "banana" might be neighbors, while "car" and "truck" are also neighbors.
*   **Different words get different addresses.** Words that have very different meanings will have addresses that are far apart. "Apple" and "car" would be very far from each other.

**Why is this useful?**

Computers don't understand words like humans do. They understand numbers. Embeddings translate words into lists of numbers (their "address"). This allows computers to:

*   **Understand relationships between words:*

## Side-by-Side Comparison

| Concept | OpenAI | Anthropic | Gemini |
|---|---|---|---|
| System Prompt | `instructions=` | `system=` | `system_instruction=` |
| Model | `model=` | `model=` | `model=` |
| Tools | `tools=` | `tools=` | `tools=` |
| Config | `dict` | `dict` | `GenerateContentConfig()` |
| User Input | `input=` | `messages=` | `contents=` |
| Chat History | Manual | Manual | Native Chat |
| Web Search | `web_search` | `web_search` | `GoogleSearch()` |




# Key Takeaways

Always structure your LLM code like this:

```python
client
MODEL
SYSTEM_PROMPT
TOOLS
CONFIG
USER_QUESTION
response
```

This keeps your code:

- readable
- reusable
- provider-independent
- easy to upgrade

The API syntax changes, but the architecture stays the same.